# Collisions and cascades: `Collision.iter_events()` and `.df`

`COLLISON.txt` records every collision of the ion and of the recoils of a full cascade, and
gets huge. pysrim2 never reads it whole: `Collision.iter_events()` streams it line by line
as flat dicts, `Collision.df` collects them chunk-wise into a DataFrame, and
`Collision.to_parquet` writes it in chunks, so memory stays small. The columns are
documented in the [API reference](https://pysrim2.readthedocs.io/en/latest/source/modules.html).

Set up SRIM once with `pysrim2 install` and check it with `pysrim2 doctor`
(see the [installation guide](https://pysrim2.readthedocs.io/en/latest/installation.html)). The notebook then finds SRIM
by itself, natively or in the container; no path is needed. SRIM-2013 only, **not** 2018.
SRIM is (c) SRIM.com (James F. Ziegler), free for non-commercial use only; see `LEGAL.rtf` next to your SRIM copy.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from srim import TRIM, Ion, Layer, Target

OUT = Path("collision-output")
OUT.mkdir(exist_ok=True)

## A full-cascade run (small)

In [ ]:
ion = Ion("Si", energy=1.0e6)  # 1 MeV

layer = Layer(
    {
        "Si": {"stoich": 0.5, "E_d": 35.0, "lattice": 0.0, "surface": 3.0},
        "C": {"stoich": 0.5, "E_d": 20.0, "lattice": 0.0, "surface": 3.0},
    },
    density=3.21,
    width=20000.0,  # 2 um
)
trim = TRIM(
    Target([layer]),
    ion,
    number_ions=5,
    calculation=2,  # full cascade
    collisions=2,  # write COLLISON.txt, including the cascades
    random_seed=12345,
)
results = trim.run()
collision = results.collision
len(collision), "ions"  # len() is the number of ions

## Stream the events

Each event is a dict. `kind` is `"ion"` for a collision of the primary ion and `"recoil"`
for a recoil of its cascade.

In [ ]:
n = {"ion": 0, "recoil": 0}
for event in collision.iter_events():
    n[event["kind"]] += 1
n

## The same as one DataFrame

In [ ]:
events = collision.df
events.head()

In [ ]:
events.groupby("kind").size()

## Electronic energy loss along the ion path

Between two successive collisions of the same ion, the energy lost per path length.

In [ ]:
ions = events[events["kind"] == "ion"].reset_index(drop=True)
same = ions["ion"].to_numpy()[1:] == ions["ion"].to_numpy()[:-1]
step = ions[["depth_A", "y_A", "z_A"]].to_numpy()
dx = np.linalg.norm(np.diff(step, axis=0), axis=1)
de = -np.diff(ions["energy_keV"].to_numpy()) * 1e3  # eV
ok = same & (dx > 0)
energy_keV = ions["energy_keV"].to_numpy()[:-1][ok]
dedx = de[ok] / dx[ok]  # eV/A

fig, ax = plt.subplots()
ax.scatter(energy_keV, dedx, s=2)
ax.set(xlabel="Ion energy [keV]", ylabel="dE/dx [eV/A]", xscale="log", title="Energy loss per step")
plt.show()

## Energy given to recoils, per atom type

In [ ]:
bins = 10 ** np.linspace(0, 6, 100)
fig, ax = plt.subplots()
for atom, group in ions.groupby("atom"):
    ax.hist(group["recoil_energy_eV"], bins=bins, histtype="step", label=atom)
ax.set(xscale="log", xlabel="Recoil energy [eV]", ylabel="collisions")
ax.legend()
plt.show()

## Defects per ion

The cascade totals are columns of the `ion` rows (`target_vacancies`, ...).

In [ ]:
ions.groupby("ion")[["target_displacements", "target_vacancies"]].sum()

## Cascade positions

Recoil rows have `depth_A`, `y_A`, `z_A` like ion rows: a map of where the damage is.

In [ ]:
recoils = events[events["kind"] == "recoil"]
fig, ax = plt.subplots()
ax.scatter(recoils["depth_A"], recoils["y_A"], s=1, alpha=0.4)
ax.set(xlabel="Depth [A]", ylabel="y [A]", title="Recoils of the cascades")
plt.show()

## Export in chunks

`chunk_rows` events are held in memory at a time (needs `pysrim2[parquet]`).

In [ ]:
rows = collision.to_parquet(OUT / "collisions.parquet", chunk_rows=10_000)
rows